## 1. Configure the full-size Kaggle run

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, random, time, math, shutil, re, unicodedata, platform, copy
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from safetensors.torch import save_file, load_file
MODE = "full"  # smoke, mini, full
STEPS, HORIZON, SEED = 400, 400, 17
PRIOR = ""  # trusted previous Kaggle output directory for this mode
ROOT = Path("/kaggle/working" if Path("/kaggle").exists() else ".") / ("Pratham-"+MODE)
if PRIOR: shutil.copytree(PRIOR, ROOT, dirs_exist_ok=True)
ROOT.mkdir(exist_ok=True,parents=True)
def write(name,obj): (ROOT/name).write_text(json.dumps(obj,indent=2,ensure_ascii=False))
def digest(b): return hashlib.sha256(b).hexdigest()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4,os.cpu_count() or 1))
device = "cuda" if torch.cuda.is_available() else "cpu"
assert MODE in ("smoke","mini","full") and 0 < STEPS <= HORIZON
if MODE=="full" and device=="cpu": raise RuntimeError("Select a GPU accelerator for the full preset")
print(platform.platform(),sys.version,torch.__version__,device,torch.version.cuda,shutil.disk_usage(ROOT).free)

## 2. Acquire the Gutenberg corpus and split by book

In [ ]:
import urllib.request, zipfile, unicodedata, collections
source="https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/gutenberg.zip"
archive=ROOT/"gutenberg.zip"
if not archive.exists():
    urllib.request.urlretrieve(source, archive)
with zipfile.ZipFile(archive) as z:
    names=sorted(n for n in z.namelist() if n.endswith(".txt"))
    docs=[unicodedata.normalize("NFC",z.read(n).decode("latin1")).strip() for n in names]
assert len(docs)>=10 and len(set(docs))==len(docs)
# Hold out a complete novel and a different author's complete stories.
validation=names.index("gutenberg/austen-emma.txt"); test=names.index("gutenberg/chesterton-brown.txt")
order=[i for i in range(len(names)) if i not in [validation,test]]+[validation,test]
docs=[docs[i] for i in order]
manifest=[dict(id=names[i],source=source,sha256=digest(doc.encode()),archive_sha256=digest(archive.read_bytes()),split="train" if j<len(docs)-2 else "validation" if j==len(docs)-2 else "test") for j,(i,doc) in enumerate(zip(order,docs))]
write("dataset_manifest.json",manifest); print("Corpus",len(docs),sum(map(len,docs)),[m["id"] for m in manifest[-2:]])

## 3. Fit the byte-level tokenizer on the training split

In [ ]:
from collections import Counter
symbols=list("banana")
for _ in range(2):
    pair=Counter(zip(symbols,symbols[1:])).most_common(1)[0][0]; merged=[]; i=0
    while i<len(symbols):
        if tuple(symbols[i:i+2])==pair: merged.append("".join(pair)); i+=2
        else: merged.append(symbols[i]); i+=1
    symbols=merged; print(pair,symbols)
assert "".join(symbols)=="banana"
target={"smoke":300,"mini":4096,"full":40478}[MODE]
tok=Tokenizer(models.BPE()); tok.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder=decoders.ByteLevel()
tok.train_from_iterator((p for doc in docs[:-2] for p in doc.splitlines() if p),trainers.BpeTrainer(vocab_size=target,min_frequency=1,special_tokens=["<eod>"],initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),show_progress=False))
tok.save(str(ROOT/"tokenizer.json"))
for s in ["", "42! café नेपाल",docs[0][:1000]]: assert tok.decode(tok.encode(s).ids)==s
V=tok.get_vocab_size(); eos=tok.token_to_id("<eod>")
streams=[torch.tensor([n for doc in group for n in tok.encode(doc).ids+[eos]]) for group in [docs[:-2],docs[-2:-1],docs[-1:]]]
print(V,[len(x) for x in streams])
assert V==40478, f"Full vocabulary expected, got {V}"

## 4. Implement the decoder-only Transformer

In [ ]:
import math, torch
from torch import nn
from torch.nn import functional as F
class Block(nn.Module):
    def __init__(self,d,h,p):
        super().__init__()
        self.h=h; self.p=p
        self.qkv=nn.Linear(d,3*d); self.out=nn.Linear(d,d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(approximate="tanh"),nn.Linear(4*d,d))
        self.norm1=nn.LayerNorm(d); self.norm2=nn.LayerNorm(d); self.drop=nn.Dropout(p)
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(x).reshape(b,t,3,self.h,d//self.h).permute(2,0,3,1,4).unbind()
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.p if self.training else 0.)
        x=self.norm1(x+self.drop(self.out(a.transpose(1,2).reshape(b,t,d))))
        return self.norm2(x+self.drop(self.ff(x)))
class Pratham(nn.Module):
    def __init__(self,v,t,d,h,l,p=.1):
        super().__init__()
        self.t=t; self.words=nn.Embedding(v,d); self.pos=nn.Embedding(t,d)
        self.blocks=nn.Sequential(*(Block(d,h,p) for _ in range(l))); self.drop=nn.Dropout(p)
        for m in self.modules():
            if isinstance(m,(nn.Linear,nn.Embedding)): nn.init.normal_(m.weight,std=.02)
            if isinstance(m,nn.Linear): nn.init.zeros_(m.bias)
    def hidden(self,x):
        if x.shape[1]>self.t: raise ValueError("Context length exceeded")
        return self.blocks(self.drop(self.words(x)+self.pos(torch.arange(x.shape[1],device=x.device))))
    def forward(self,x): return F.linear(self.hidden(x),self.words.weight)

## 5. Validate causality, parameter count and optimization

In [ ]:
T,D,H,L={"smoke":(16,48,4,2),"mini":(128,256,4,4),"full":(512,768,12,12)}[MODE]
config=dict(v=V,t=T,d=D,h=H,l=L); write("config.json",config)
def batch(stream,step,b=1,t=T):
    g=torch.Generator().manual_seed(SEED+step)
    start=torch.randint(len(stream)-t,(b,),generator=g)
    z=stream[start[:,None]+torch.arange(t+1)]
    return z[:,:-1].to(device),z[:,1:].to(device)
def loss(net,x,y): return F.cross_entropy(net(x).flatten(0,1),y.flatten())
probe=Pratham(V,16,48,4,2,0).to(device); x,y=batch(streams[0],0,t=16)
assert torch.equal(x[:,1:],y[:,:-1])
changed=x.clone(); changed[:,-1]=(changed[:,-1]+1)%V
torch.testing.assert_close(probe(x)[:,:-1],probe(changed)[:,:-1])
opt=torch.optim.AdamW(probe.parameters(),lr=.005)
initial=loss(probe,x,y).item()
for _ in range(30):
    opt.zero_grad(); loss(probe,x,y).backward(); opt.step()
end=loss(probe,x,y).item(); assert end<.7*initial,(initial,end)
del probe,opt
model=Pratham(**config).to(device)
params=sum(p.numel() for p in model.parameters())
assert params==(V+T)*D+L*(12*D*D+13*D)
print("Fixed batch loss",initial,end,"parameters",params)

## 6. Train with gradient accumulation and resumable checkpoints

In [ ]:
from tqdm.auto import trange
B,A=(1,4) if MODE=="full" else (4,1)
lr=.00025 if MODE=="full" else .001
optimizer=torch.optim.AdamW(model.parameters(),lr=lr)
scaler=torch.amp.GradScaler("cuda",enabled=device=="cuda")
signature=digest(json.dumps([config,manifest,HORIZON,SEED,B,A,lr,device,digest((ROOT/"tokenizer.json").read_bytes())],sort_keys=True).encode())
def restore(path):
    ck=torch.load(path,map_location="cpu",weights_only=True)
    if ck["signature"]!=signature: raise ValueError("Checkpoint config/data/tokenizer changed")
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
    torch.set_rng_state(ck["rng"])
    if device=="cuda": torch.cuda.set_rng_state(ck["cuda"])
    return ck["step"],ck["history"]
paths=sorted(ROOT.glob("checkpoint-v2-*.pt"),key=lambda p:p.stat().st_mtime_ns,reverse=True)
start,history=0,[]
for path in paths:
    try: start,history=restore(path); break
    except ValueError: raise
    except Exception as exc: print("Unreadable checkpoint",path,exc)
else:
    if paths: raise RuntimeError("Restore a trusted previous notebook output")
@torch.no_grad()
def evaluate(net,data):
    net.eval(); totals=np.zeros(4)
    for j in range(0,min(len(data)-1,8192),T):
        block=data[j:min(j+T+1,len(data),8193)].to(device)
        if len(block)<2: continue
        z=net(block[:-1][None])[0]; y=block[1:]
        totals += [F.cross_entropy(z,y,reduction="sum").item(),(z.argmax(-1)==y).sum().item(),(z.topk(5).indices==y[:,None]).any(-1).sum().item(),len(y)]
    ce,acc,top5=map(float,totals[:3]/totals[3])
    return dict(loss=ce,perplexity=math.exp(ce),accuracy=acc,top5=top5,tokens=int(totals[3]))
for step in trange(start+1,STEPS+1):
    tick=time.monotonic(); model.train(); optimizer.zero_grad(set_to_none=True)
    warm=min(2000,max(1,HORIZON//10))
    rate=lr*(step/warm if step<=warm else .5*(1+math.cos(math.pi*(step-warm)/(HORIZON-warm))))
    for pg in optimizer.param_groups: pg["lr"]=rate
    ce=0.
    for a in range(A):
        x,y=batch(streams[0],step*A+a,B)
        with torch.autocast(device_type=device,dtype=torch.float16,enabled=device=="cuda"):
            value=loss(model,x,y)
        if not torch.isfinite(value): raise FloatingPointError("Nonfinite training loss")
        scaler.scale(value/A).backward(); ce+=value.item()/A
    scaler.unscale_(optimizer); norm=nn.utils.clip_grad_norm_(model.parameters(),1.).item()
    scaler.step(optimizer); scaler.update()
    history.append(dict(step=step,loss=ce,lr=rate,gradient_norm=norm,tokens=step*B*A*T,tokens_per_second=B*A*T/(time.monotonic()-tick)))
    if step%100==0 or step==STEPS:
        history[-1]["validation"]=evaluate(model,streams[1])
        state=dict(signature=signature,step=step,model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),rng=torch.get_rng_state(),cuda=torch.cuda.get_rng_state() if device=="cuda" else None,history=history)
        path=ROOT/f"checkpoint-v2-{(step//100)%2}.pt"; tmp=path.with_suffix(".tmp")
        torch.save(state,tmp); os.replace(tmp,path)
        assert restore(path)[0]==step
write("training_metrics.json",history)
results=evaluate(model,streams[2]); write("evaluation_results.json",results)
print("Final update",history[-1],"Held-out",results)

## 7. Export base weights and sample continuations

In [ ]:
from safetensors.torch import save_file,load_file
save_file({k:v.detach().cpu().contiguous() for k,v in model.state_dict().items()},str(ROOT/"model.safetensors"))
fresh=Pratham(**json.loads((ROOT/"config.json").read_text())).to(device).eval()
fresh.load_state_dict(load_file(str(ROOT/"model.safetensors")))
model.eval()
with torch.no_grad(): torch.testing.assert_close(fresh(x),model(x))
@torch.no_grad()
def generate(net,tok,prompt,limit=32,temp=.8,k=20,p=.95,seed=17):
    dev=next(net.parameters()).device; eos=tok.token_to_id("<eod>")
    ids=tok.encode(prompt).ids or [eos]; rng=torch.Generator(device=dev).manual_seed(seed)
    for _ in range(limit):
        z=net(torch.tensor([ids[-net.t:]],device=dev))[0,-1]
        if temp==0: token=int(z.argmax())
        else:
            vals,indices=(z/temp).topk(min(k or len(z),len(z)))
            probs=vals.softmax(-1); probs=probs.masked_fill(probs.cumsum(0)-probs>=p,0)
            token=int(indices[torch.multinomial(probs,1,generator=rng)].item())
        if token==eos: break
        ids.append(token)
    return tok.decode(ids)
samples=[generate(fresh,tok,s) for s in ["Once upon a time","The sun rises in","In the middle of the night"]]
write("samples.json",samples); print(*samples,sep="\n")
(ROOT/'inference.py').write_text("import math, torch\nfrom torch import nn\nfrom torch.nn import functional as F\nclass Block(nn.Module):\n    def __init__(self,d,h,p):\n        super().__init__()\n        self.h=h; self.p=p\n        self.qkv=nn.Linear(d,3*d); self.out=nn.Linear(d,d)\n        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(approximate=\"tanh\"),nn.Linear(4*d,d))\n        self.norm1=nn.LayerNorm(d); self.norm2=nn.LayerNorm(d); self.drop=nn.Dropout(p)\n    def forward(self,x):\n        b,t,d=x.shape\n        q,k,v=self.qkv(x).reshape(b,t,3,self.h,d//self.h).permute(2,0,3,1,4).unbind()\n        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.p if self.training else 0.)\n        x=self.norm1(x+self.drop(self.out(a.transpose(1,2).reshape(b,t,d))))\n        return self.norm2(x+self.drop(self.ff(x)))\nclass Pratham(nn.Module):\n    def __init__(self,v,t,d,h,l,p=.1):\n        super().__init__()\n        self.t=t; self.words=nn.Embedding(v,d); self.pos=nn.Embedding(t,d)\n        self.blocks=nn.Sequential(*(Block(d,h,p) for _ in range(l))); self.drop=nn.Dropout(p)\n        for m in self.modules():\n            if isinstance(m,(nn.Linear,nn.Embedding)): nn.init.normal_(m.weight,std=.02)\n            if isinstance(m,nn.Linear): nn.init.zeros_(m.bias)\n    def hidden(self,x):\n        if x.shape[1]>self.t: raise ValueError(\"Context length exceeded\")\n        return self.blocks(self.drop(self.words(x)+self.pos(torch.arange(x.shape[1],device=x.device))))\n    def forward(self,x): return F.linear(self.hidden(x),self.words.weight)\n@torch.no_grad()\ndef generate(net,tok,prompt,limit=32,temp=.8,k=20,p=.95,seed=17):\n    dev=next(net.parameters()).device; eos=tok.token_to_id(\"<eod>\")\n    ids=tok.encode(prompt).ids or [eos]; rng=torch.Generator(device=dev).manual_seed(seed)\n    for _ in range(limit):\n        z=net(torch.tensor([ids[-net.t:]],device=dev))[0,-1]\n        if temp==0: token=int(z.argmax())\n        else:\n            vals,indices=(z/temp).topk(min(k or len(z),len(z)))\n            probs=vals.softmax(-1); probs=probs.masked_fill(probs.cumsum(0)-probs>=p,0)\n            token=int(indices[torch.multinomial(probs,1,generator=rng)].item())\n        if token==eos: break\n        ids.append(token)\n    return tok.decode(ids)\n\nif __name__==\"__main__\":\n    import json,sys\n    from pathlib import Path\n    from tokenizers import Tokenizer\n    from safetensors.torch import load_file\n    root=Path(__file__).resolve().parent\n    net=Pratham(**json.loads((root/\"config.json\").read_text())).eval()\n    net.load_state_dict(load_file(str(root/\"model.safetensors\")))\n    print(generate(net,Tokenizer.from_file(str(root/\"tokenizer.json\")),\" \".join(sys.argv[1:])))\n")


## 8. Instruction-tune and evaluate the assistant

In [ ]:
# Assistant-token supervised fine-tuning, separate validation and release export.
import gc, json, random, shutil
from huggingface_hub import hf_hub_download
from safetensors.torch import save_file,load_file
del optimizer, scaler, model
if 'state' in globals(): del state
gc.collect(); torch.cuda.empty_cache()
path=hf_hub_download("databricks/databricks-dolly-15k","databricks-dolly-15k.jsonl",repo_type="dataset")
rows=[json.loads(line) for line in open(path)]
random.Random(SEED).shuffle(rows)
def encode_row(row):
    question=row["instruction"]+("\n"+row["context"] if row["context"] else "")
    prefix=tok.encode("User: "+question+"\nAssistant:").ids
    answer=tok.encode(" "+row["response"]).ids+[eos]
    if len(prefix)>320 or len(answer)<2: return None
    seq=(prefix+answer)[:T+1]
    labels=[-100]*(len(prefix)-1)+seq[len(prefix):]
    return torch.tensor([seq[:-1]],device=device),torch.tensor([labels],device=device)
usable=[row for row in rows if len(row["instruction"])+len(row["context"])<900]
train=usable[100:1124]; validation=usable[:100]
def answer_loss(row):
    pair=encode_row(row)
    if pair is None: return None
    x,y=pair
    with torch.autocast("cuda",dtype=torch.float16): return F.cross_entropy(fresh(x).flatten(0,1),y.flatten(),ignore_index=-100)
@torch.no_grad()
def validate():
    fresh.eval(); values=[v.item() for row in validation if (v:=answer_loss(row)) is not None]
    return sum(values)/len(values)
before=validate(); fresh.train()
sopt=torch.optim.AdamW(fresh.parameters(),lr=2e-5)
ss=torch.amp.GradScaler("cuda"); used=0
for row in train:
    v=answer_loss(row)
    if v is None: continue
    if not torch.isfinite(v): raise FloatingPointError("SFT loss")
    sopt.zero_grad(set_to_none=True); ss.scale(v).backward(); ss.unscale_(sopt)
    torch.nn.utils.clip_grad_norm_(fresh.parameters(),1.0); ss.step(sopt); ss.update(); used+=1
after=validate()
release=ROOT/"chat-release"; release.mkdir(exist_ok=True)
for name in ["config.json","tokenizer.json","inference.py","dataset_manifest.json","training_metrics.json","evaluation_results.json"]:
    shutil.copy2(ROOT/name,release/name)
save_file({k:v.detach().cpu().contiguous() for k,v in fresh.state_dict().items()},str(release/"model.safetensors"))
reloaded=Pratham(**config).eval()
reloaded.load_state_dict(load_file(str(release/"model.safetensors")))
probe_ids=torch.tensor([[eos,1,2]])
with torch.no_grad(): torch.testing.assert_close(reloaded(probe_ids),fresh(probe_ids.cuda()).cpu(),atol=.002,rtol=.002)
samples={q:generate(reloaded,tok,"User: "+q+"\nAssistant:",limit=32) for q in ["Hello!","What is two plus two?","What makes rain fall?"]}
summary=dict(parameters=params,base_tokens=history[-1]["tokens"],base_test=results,sft_examples=used,sft_before=before,sft_after=after,samples=samples,dolly_sha256=digest(Path(path).read_bytes()))
(release/"release_metrics.json").write_text(json.dumps(summary,indent=2))
(release/"README.md").write_text("---\nlanguage: en\nlicense: cc-by-sa-3.0\ntags: [text-generation, from-scratch, experimental]\n---\n# Pratham-117M\nAn experimental English GPT-1-inspired model by Rahul Chaube. Learned absolute positions, post-LayerNorm, tied embeddings, 12 layers and 12 heads. The tokenizer and base weights were learned from scratch. Corpus source and document-level splits are in dataset_manifest.json. Supervised adaptation uses Databricks Dolly 15k (Databricks, CC BY-SA 3.0), transformed into role-formatted sequences with loss over responses. See https://huggingface.co/datasets/databricks/databricks-dolly-15k and https://creativecommons.org/licenses/by-sa/3.0/. Input is English text. Responses may be incoherent or inaccurate. Read inference.py before running custom Python.\n\n## Measured run\n```json\n"+json.dumps(summary,indent=2)+"\n```\n")
print("CHAT_RELEASE_EXPORTED",summary)

## 9. Publish the model and its pinned Gradio Space

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
token=UserSecretsClient().get_secret("HF_TOKEN")
api=HfApi(token=token)
assert api.whoami()["name"]=="oncody","Expected oncody account"
model_id="oncody/Pratham-117M"; space_id="oncody/Pratham-117M-Chat"
api.create_repo(model_id,repo_type="model",private=False,exist_ok=True)
commit=api.upload_folder(repo_id=model_id,folder_path=release,commit_message="Release measured full-size base plus Dolly adaptation")

space_id="oncody/Pratham-117M-Chat"
import sys, subprocess, importlib.util
if any(importlib.util.find_spec(x) is None for x in ['onnx','onnxruntime']):
    subprocess.check_call([sys.executable,'-m','pip','-q','install','onnx','onnxruntime'])
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic,QuantType
class BrowserLM(nn.Module):
    def forward(self,ids):
        h=reloaded.words(ids)+reloaded.pos(torch.arange(ids.shape[1]))
        for block in reloaded.blocks:
            b,t,d=h.shape
            q,k,v=block.qkv(h).reshape(b,t,3,block.h,d//block.h).permute(2,0,3,1,4).unbind()
            scores=q@k.transpose(-2,-1)/math.sqrt(d//block.h)
            pos=torch.arange(t)
            w=scores.masked_fill(pos[:,None]<pos[None,:],-1e4).softmax(-1)
            h=block.norm1(h+block.out((w@v).transpose(1,2).reshape(b,t,d)))
            h=block.norm2(h+block.ff(h))
        return F.linear(h[:,-1],reloaded.words.weight)
wrap=BrowserLM();wrap.core=reloaded.eval()
tmp=ROOT/'browser-fp32.onnx'; dest=ROOT/'chat.int8.onnx'
torch.onnx.export(wrap.eval(),(probe_ids,),str(tmp),input_names=['ids'],output_names=['scores'],dynamic_axes={'ids':{1:'time'}},opset_version=17,dynamo=False)
quantize_dynamic(str(tmp),str(dest),weight_type=QuantType.QInt8,op_types_to_quantize=['MatMul','Gather','Gemm'])
sess=ort.InferenceSession(str(dest),providers=['CPUExecutionProvider'])
for length in [3,11]:
    sample=torch.arange(length)[None]
    y=sess.run(None,{'ids':sample.numpy()})[0]
    with torch.no_grad(): ref=reloaded(sample)[:,-1].numpy()
    cosine=float(np.sum(y*ref)/(np.linalg.norm(y)*np.linalg.norm(ref)))
    assert cosine>.98,cosine
    print('ONNX verified',length,cosine)
rev=api.upload_file(path_or_fileobj=str(dest),path_in_repo=dest.name,repo_id=model_id,commit_message='Add verified browser inference export').oid
static=ROOT/'static-space';static.mkdir(exist_ok=True)
(static/'README.md').write_text('---\ntitle: Pratham-117M Chat\nsdk: static\napp_file: index.html\n---\nExperimental client-side chat with the independently trained Pratham model.\n')
(static/'index.html').write_text("<!doctype html><html lang=\"en\"><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\"><title>Pratham-117M Chat</title><style>body{max-width:720px;margin:7vh auto;padding:0 20px;font:17px/1.5 system-ui;background:#faf9f5;color:#172622}h1{font-size:2.3em}small{color:#495e54}textarea{width:100%;box-sizing:border-box;padding:14px;font:inherit}button{background:#17573e;color:white;border:0;padding:10px 22px;border-radius:6px}button:disabled{opacity:.5}p{white-space:pre-wrap}#status{margin:18px 0}</style><body data-base=\"BASEURL\"><h1>Pratham-117M Chat</h1><small>Experimental English model by Rahul Chaube. Replies are generated on your device and can be inaccurate or incoherent.</small><div id=\"status\" role=\"status\">Starting…</div><div id=\"log\" aria-live=\"polite\"></div><form><label>Your message<textarea required rows=\"2\"></textarea></label><button disabled>Send</button></form><script type=\"module\" src=\"./app.js\"></script></body></html>".replace('BASEURL',f'https://huggingface.co/{model_id}/resolve/{rev}/'))
(static/'app.js').write_text("import { Tokenizer } from 'https://cdn.jsdelivr.net/npm/@huggingface/tokenizers@0.1.3/+esm';\nimport * as ort from 'https://cdn.jsdelivr.net/npm/onnxruntime-web@1.22.0/dist/ort.wasm.min.mjs';\nort.env.wasm.numThreads=1;\nort.env.wasm.wasmPaths='https://cdn.jsdelivr.net/npm/onnxruntime-web@1.22.0/dist/';\nconst base=document.body.dataset.base;\nconst log=document.querySelector('#log'), status=document.querySelector('#status'),send=document.querySelector('button'),box=document.querySelector('textarea');\nlet session,tokenizer,history='';\nfunction line(who,msg){const row=document.createElement('p');const label=document.createElement('strong');label.textContent=who+' ';row.append(label,document.createTextNode(msg));log.append(row);return row}\nasync function init(){status.textContent='Loading tokenizer and model (~148 MB on first use)…';\nconst r=await fetch(base+'tokenizer.json');if(!r.ok)throw Error('Tokenizer download failed');\ntokenizer=new Tokenizer(await r.json(),{});session=await ort.InferenceSession.create(base+'chat.int8.onnx',{executionProviders:['wasm']});status.textContent='Ready';send.disabled=false}\ndocument.querySelector('form').onsubmit=async(e)=>{e.preventDefault();const q=box.value.trim();if(!q||send.disabled)return;send.disabled=true;box.value='';line('You',q);const row=line('Pratham','…');let out=[];\ntry{const prompt=history+'User: '+q+'\\nAssistant:';let ids=Array.from(tokenizer.encode(prompt).ids,Number).slice(-128);for(let i=0;i<32;i++){const {scores}=await session.run({ids:new ort.Tensor('int64',BigInt64Array.from(ids,BigInt),[1,ids.length])});const z=scores.data;const top=Array.from(z.keys()).sort((a,b)=>z[b]-z[a]).slice(0,20);let weights=top.map(j=>Math.exp((z[j]-z[top[0]])/.8));let t=Math.random()*weights.reduce((a,b)=>a+b,0),j=0;while(j<top.length-1&&(t-=weights[j])>0)j++;const next=top[j];if(next===0)break;out.push(next);ids.push(next);row.textContent='Pratham  '+tokenizer.decode(out);await new Promise(resolve=>setTimeout(resolve,0))}const reply=tokenizer.decode(out).split('\\nUser:')[0].trim();history=prompt+' '+reply+'\\n';row.textContent='Pratham  '+(reply||'[End of reply]');status.textContent='Ready'}catch(err){row.textContent='Generation failed: '+err.message}finally{send.disabled=false}};\ninit().catch(err=>status.textContent='Loading failed: '+err.message);\n")
api.create_repo(space_id,repo_type='space',space_sdk='static',private=False,exist_ok=True)
api.upload_folder(repo_id=space_id,repo_type='space',folder_path=static,commit_message='Serve client-side Pratham inference')
print('STATIC_PUBLISHED',rev,dest.stat().st_size)
